# Exercice 14 - Fonctions SQL avec Spark

## Objectifs
- Utiliser Spark SQL pour interroger les DataFrames
- Maitriser les fonctions SQL courantes
- Combiner SQL et API DataFrame
- Creer des vues temporaires et permanentes

---

## 1. Spark SQL

```
+------------------------------------------------------------------+
|                         SPARK SQL                                |
+------------------------------------------------------------------+
|                                                                  |
|  DataFrame     <------>    Vue SQL    <------>    Requete SQL    |
|                                                                  |
|  df_orders     ------>    orders      ------>    SELECT * FROM   |
|                           (temp view)            orders          |
|                                                                  |
+------------------------------------------------------------------+
|                                                                  |
|  Avantages SQL :                                                 |
|  - Syntaxe familiere                                             |
|  - Requetes complexes lisibles                                   |
|  - Meme optimiseur que DataFrame API                             |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Spark SQL") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

print("Spark pret")

In [ ]:
# Charger les donnees
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

tables = ["orders", "order_details", "products", "customers", "employees", "categories", "suppliers"]

for table in tables:
    df = spark.read.jdbc(url=jdbc_url, table=table, properties=jdbc_properties)
    df.createOrReplaceTempView(table)  # Creer une vue SQL
    print(f"[OK] {table} : {df.count()} lignes")

print("\nToutes les vues SQL creees")

## 3. Requetes SQL de base

In [ ]:
# SELECT simple
spark.sql("""
    SELECT customer_id, company_name, country
    FROM customers
    LIMIT 10
""").show()

In [ ]:
# WHERE
spark.sql("""
    SELECT product_name, unit_price
    FROM products
    WHERE unit_price > 50
    ORDER BY unit_price DESC
""").show()

In [ ]:
# GROUP BY
spark.sql("""
    SELECT country, COUNT(*) as nb_clients
    FROM customers
    GROUP BY country
    ORDER BY nb_clients DESC
""").show()

In [ ]:
# HAVING
spark.sql("""
    SELECT country, COUNT(*) as nb_clients
    FROM customers
    GROUP BY country
    HAVING COUNT(*) >= 5
    ORDER BY nb_clients DESC
""").show()

## 4. Jointures en SQL

In [ ]:
# INNER JOIN
spark.sql("""
    SELECT 
        o.order_id,
        o.order_date,
        c.company_name,
        c.country
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    LIMIT 10
""").show()

In [ ]:
# Jointure multiple
spark.sql("""
    SELECT 
        o.order_id,
        c.company_name,
        p.product_name,
        cat.category_name,
        od.quantity,
        od.unit_price
    FROM order_details od
    JOIN orders o ON od.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON od.product_id = p.product_id
    JOIN categories cat ON p.category_id = cat.category_id
    LIMIT 10
""").show(truncate=False)

In [ ]:
# LEFT JOIN avec condition
spark.sql("""
    SELECT 
        c.customer_id,
        c.company_name,
        COUNT(o.order_id) as nb_commandes
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.company_name
    ORDER BY nb_commandes
    LIMIT 10
""").show()

## 5. Fonctions de date

In [ ]:
# Extraction de composants de date
spark.sql("""
    SELECT 
        order_id,
        order_date,
        YEAR(order_date) as annee,
        MONTH(order_date) as mois,
        DAY(order_date) as jour,
        QUARTER(order_date) as trimestre,
        DAYOFWEEK(order_date) as jour_semaine,
        WEEKOFYEAR(order_date) as semaine
    FROM orders
    LIMIT 5
""").show()

In [ ]:
# Calcul de differences de dates
spark.sql("""
    SELECT 
        order_id,
        order_date,
        shipped_date,
        DATEDIFF(shipped_date, order_date) as delai_jours
    FROM orders
    WHERE shipped_date IS NOT NULL
    LIMIT 10
""").show()

In [ ]:
# Formatage de dates
spark.sql("""
    SELECT 
        order_id,
        order_date,
        DATE_FORMAT(order_date, 'dd/MM/yyyy') as date_fr,
        DATE_FORMAT(order_date, 'MMMM yyyy') as mois_annee,
        DATE_FORMAT(order_date, 'EEEE') as nom_jour
    FROM orders
    LIMIT 5
""").show(truncate=False)

## 6. Fonctions de chaines

In [ ]:
# Manipulation de chaines
spark.sql("""
    SELECT 
        company_name,
        UPPER(company_name) as upper_name,
        LOWER(company_name) as lower_name,
        LENGTH(company_name) as longueur,
        SUBSTRING(company_name, 1, 10) as debut
    FROM customers
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Concatenation et remplacement
spark.sql("""
    SELECT 
        first_name,
        last_name,
        CONCAT(first_name, ' ', last_name) as full_name,
        CONCAT_WS(', ', last_name, first_name) as nom_formel
    FROM employees
""").show()

In [ ]:
# Recherche et pattern matching
spark.sql("""
    SELECT product_name
    FROM products
    WHERE product_name LIKE '%Sauce%'
       OR product_name LIKE '%Syrup%'
""").show(truncate=False)

In [ ]:
# Expression reguliere
spark.sql("""
    SELECT product_name
    FROM products
    WHERE product_name RLIKE '^[A-C]'
""").show(truncate=False)

## 7. Fonctions d'agregation

In [ ]:
# Agregations de base
spark.sql("""
    SELECT 
        COUNT(*) as nb_produits,
        ROUND(AVG(unit_price), 2) as prix_moyen,
        MIN(unit_price) as prix_min,
        MAX(unit_price) as prix_max,
        ROUND(SUM(units_in_stock), 0) as stock_total
    FROM products
""").show()

In [ ]:
# Agregation avec GROUP BY
spark.sql("""
    SELECT 
        cat.category_name,
        COUNT(*) as nb_produits,
        ROUND(AVG(p.unit_price), 2) as prix_moyen,
        SUM(p.units_in_stock) as stock_total
    FROM products p
    JOIN categories cat ON p.category_id = cat.category_id
    GROUP BY cat.category_name
    ORDER BY nb_produits DESC
""").show()

In [ ]:
# Agregation conditionnelle
spark.sql("""
    SELECT 
        category_name,
        COUNT(*) as total_produits,
        SUM(CASE WHEN unit_price < 20 THEN 1 ELSE 0 END) as produits_pas_chers,
        SUM(CASE WHEN unit_price >= 20 AND unit_price < 50 THEN 1 ELSE 0 END) as produits_moyens,
        SUM(CASE WHEN unit_price >= 50 THEN 1 ELSE 0 END) as produits_chers
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY category_name
""").show()

## 8. Sous-requetes

In [ ]:
# Sous-requete dans WHERE
spark.sql("""
    SELECT product_name, unit_price
    FROM products
    WHERE unit_price > (
        SELECT AVG(unit_price) FROM products
    )
    ORDER BY unit_price DESC
""").show()

In [ ]:
# Sous-requete avec IN
spark.sql("""
    SELECT company_name, country
    FROM customers
    WHERE customer_id IN (
        SELECT DISTINCT customer_id
        FROM orders
        WHERE order_date >= '1997-01-01'
    )
""").show()

In [ ]:
# Sous-requete avec EXISTS
spark.sql("""
    SELECT c.company_name
    FROM customers c
    WHERE EXISTS (
        SELECT 1
        FROM orders o
        JOIN order_details od ON o.order_id = od.order_id
        WHERE o.customer_id = c.customer_id
        AND od.quantity >= 100
    )
""").show(truncate=False)

## 9. Common Table Expressions (CTE)

In [ ]:
# CTE simple
spark.sql("""
    WITH ventes_client AS (
        SELECT 
            c.customer_id,
            c.company_name,
            SUM(od.unit_price * od.quantity) as ca_total
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_details od ON o.order_id = od.order_id
        GROUP BY c.customer_id, c.company_name
    )
    SELECT *
    FROM ventes_client
    WHERE ca_total > 30000
    ORDER BY ca_total DESC
""").show(truncate=False)

In [ ]:
# CTE multiples
spark.sql("""
    WITH 
    ventes_mensuelles AS (
        SELECT 
            YEAR(order_date) as annee,
            MONTH(order_date) as mois,
            SUM(od.unit_price * od.quantity) as ca
        FROM orders o
        JOIN order_details od ON o.order_id = od.order_id
        GROUP BY YEAR(order_date), MONTH(order_date)
    ),
    moyenne AS (
        SELECT AVG(ca) as ca_moyen
        FROM ventes_mensuelles
    )
    SELECT 
        v.annee,
        v.mois,
        ROUND(v.ca, 2) as ca,
        ROUND(m.ca_moyen, 2) as ca_moyen,
        CASE 
            WHEN v.ca > m.ca_moyen THEN 'Au dessus'
            ELSE 'En dessous'
        END as performance
    FROM ventes_mensuelles v
    CROSS JOIN moyenne m
    ORDER BY v.annee, v.mois
""").show()

## 10. Fonctions de fenetre (Window Functions)

In [ ]:
# ROW_NUMBER, RANK, DENSE_RANK
spark.sql("""
    SELECT 
        category_name,
        product_name,
        unit_price,
        ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY unit_price DESC) as row_num,
        RANK() OVER (PARTITION BY category_name ORDER BY unit_price DESC) as rang,
        DENSE_RANK() OVER (PARTITION BY category_name ORDER BY unit_price DESC) as dense_rang
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
""").show(20)

In [ ]:
# LAG et LEAD
spark.sql("""
    WITH ventes_mois AS (
        SELECT 
            DATE_TRUNC('month', order_date) as mois,
            SUM(od.unit_price * od.quantity) as ca
        FROM orders o
        JOIN order_details od ON o.order_id = od.order_id
        GROUP BY DATE_TRUNC('month', order_date)
    )
    SELECT 
        mois,
        ROUND(ca, 2) as ca_actuel,
        ROUND(LAG(ca, 1) OVER (ORDER BY mois), 2) as ca_precedent,
        ROUND(LEAD(ca, 1) OVER (ORDER BY mois), 2) as ca_suivant,
        ROUND((ca - LAG(ca, 1) OVER (ORDER BY mois)) / LAG(ca, 1) OVER (ORDER BY mois) * 100, 2) as evolution_pct
    FROM ventes_mois
    ORDER BY mois
""").show()

In [ ]:
# Cumul et moyenne mobile
spark.sql("""
    WITH ventes_jour AS (
        SELECT 
            order_date,
            SUM(od.unit_price * od.quantity) as ca
        FROM orders o
        JOIN order_details od ON o.order_id = od.order_id
        GROUP BY order_date
    )
    SELECT 
        order_date,
        ROUND(ca, 2) as ca,
        ROUND(SUM(ca) OVER (ORDER BY order_date), 2) as ca_cumule,
        ROUND(AVG(ca) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) as moyenne_7j
    FROM ventes_jour
    ORDER BY order_date
    LIMIT 20
""").show()

---

## Exercice

**Objectif** : Ecrire des requetes SQL complexes

**Consigne** :
1. Trouvez le top 3 des produits par categorie (en termes de CA)
2. Calculez le CA mensuel avec le cumul annuel
3. Trouvez les clients qui ont commande plus que la moyenne

A vous de jouer :

In [ ]:
# TODO: Top 3 produits par categorie

In [ ]:
# TODO: CA mensuel avec cumul annuel

In [ ]:
# TODO: Clients au dessus de la moyenne

---

## Resume

Dans ce notebook, vous avez appris :
- Comment utiliser **Spark SQL** avec des vues temporaires
- Les **fonctions de date** (YEAR, MONTH, DATEDIFF, DATE_FORMAT)
- Les **fonctions de chaines** (CONCAT, UPPER, LIKE, RLIKE)
- Les **fonctions d'agregation** (SUM, AVG, COUNT, CASE)
- Les **sous-requetes** et **CTE**
- Les **fonctions de fenetre** (ROW_NUMBER, LAG, LEAD, cumul)

### Prochaine etape
Dans le prochain notebook, nous apprendrons les bases de Kafka.